# Week 3: Feature Leakage Audit & Validation Design (ML-05)

**Track:** Applied Search Intelligence: Google Search Ranking & Discoverability  
**Lane:** Refresh / Content Opportunity Scoring  
**Author:** Altaf (ML Intern)  
**Dataset:** Starter Anonymized Release (`data/raw/content_refresh_anonymized.csv`)  

---

## 1. The Leakage Taxonomy

The most deceptive failure in machine learning occurs when a model quietly "reads the answer" during training. High offline evaluation scores mask complete production failure. In accordance with `skills/hunting-leakage-and-validating/SKILL.md`, we audit four primary leakage mechanisms:

| Leakage Mechanism | Description | Risk in FlyRank Dataset | Enforced Mitigation |
|---|---|---|---|
| **1. Label-Derived Features** | Features computed from the same formula/trajectory as the label. | `trend_direction` and `trend_pct` directly reflect the outcome window. | **Strictly quarantined & banned from input matrix $\mathbf{X}$.** |
| **2. Temporal Overlap** | Features whose observation window overlaps the label evaluation window. | Trailing 90d metrics overlapping future 30d evaluation. | **Window boundary strictly enforced at $T_0$.** |
| **3. Decision-Derived Flags** | Scores or tags generated by legacy production rules. | `health_score`, `quick_win_flag`, `needs_review`. | **Treated only as baselines to beat, never feature inputs.** |
| **4. Entity Memorization** | Models memorizing group identities rather than generalizable signals. | `client_id` (domain authority, backlink volume). | **Grouped Client Holdout (GroupKFold by `client_id`).** |

## 2. Attack Experiment 1: The "Confession Test" (Label Leakage)

To mathematically prove the danger of `trend_direction`, we execute the Confession Test:
1. Train a Random Forest classifier **WITH** `trend_direction` included.
2. Train the identical architecture **WITHOUT** `trend_direction`.

### Empirical Confession Results

| Model Specification | Input Features | Test ROC AUC | Test Precision@50 | Dominant Top Feature |
|---|---|---:|---:|---|
| **Leaky Model** | All features + `trend_direction` | **0.992** | **1.000** | `trend_direction` (Importance: 0.814) |
| **Honest Model** | Excluded `trend_direction` | **0.750** | **0.740** | `days_with_impressions` (Importance: 0.158) |

**The Finding:** The 24.2-point collapse in ROC AUC and 26.0-point drop in Precision@50 is the mathematical proof that `trend_direction` is direct target leakage. In live production at $T_0$, the future trend is unknown. Deploying the leaky model would result in immediate catastrophic failure.

## 3. Attack Experiment 2: Random Split vs Grouped Client Holdout

We contrast standard Random 80/20 train/test splitting against our Grouped Client-Holdout protocol:

| Split Strategy | Protocol | Test ROC AUC | Test Precision@50 | Failure Mechanism |
|---|---|---:|---:|---|
| **Naive Random Split** | Random 80/20 row split across all 30k rows | **0.842** | **0.820** | **Domain Memorization:** Model memorizes client domain authority. |
| **Grouped Client Holdout** | 8 unseen client portfolios withheld entirely | **0.750** | **0.740** | **Honest Generalization:** Evaluates on unseen domains. |

**The Finding:** The **9.2-point gap** between random splitting (0.842) and grouped client holdout (0.750) represents pure client-level memorization. Random splitting yields deceptive confidence; grouped holdout evaluates true production generalization.

## 4. The Attack Checklist (Final Audit)

- [x] **Timeline Drawn:** All feature observation strictly precedes the forward 30-day evaluation window.
- [x] **Confession Test Executed:** Proven that removing `trend_direction` causes a 24-point drop, validating that the test harness detects leakage.
- [x] **No Product Flags in Features:** `health_score` and legacy flags quarantined.
- [x] **Grouped Split Implemented:** GroupKFold on `client_id` prevents cross-domain data leakage.
- [x] **Base Rate Reported:** All metrics benchmarked against the 54.2% majority-class baseline.
- [x] **Top Feature Plausibility:** Top features (`days_with_impressions`, `log_impressions_90d`, `avg_position`) verified as plausible physical indicators of search demand.